
7.2 Read Silver data
Cell 1:

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table(
    "ask_sarthak.silver.gl_transactions"
)

print("Silver records:", silver_df.count())


7.3 Create the Company dimension

In [0]:
dim_company = (
    silver_df
    .select("company_code")
    .distinct()
    .withColumn(
        "company_name",
        F.when(F.col("company_code") == "1000", "India Operations")
         .when(F.col("company_code") == "2000", "North Operations")
         .when(F.col("company_code") == "3000", "South Operations")
         .when(F.col("company_code") == "4000", "West Operations")
         .otherwise("International Operations")
    )
    .withColumn("country", F.lit("India"))
)

display(dim_company)


7.4 Create the GL Account dimension

In [0]:
dim_gl_account = (
    silver_df
    .select(
        "gl_account",
        "account_name",
        "account_type"
    )
    .distinct()
)

display(dim_gl_account)


7.5 Create Vendor and Customer dimensions

In [0]:
dim_vendor = (
    silver_df
    .filter(F.col("vendor_id").isNotNull())
    .select("vendor_id")
    .distinct()
    .withColumn(
        "vendor_name",
        F.concat(F.lit("Vendor "), F.col("vendor_id"))
    )
)

dim_customer = (
    silver_df
    .filter(F.col("customer_id").isNotNull())
    .select("customer_id")
    .distinct()
    .withColumn(
        "customer_name",
        F.concat(F.lit("Customer "), F.col("customer_id"))
    )
)


7.6 Create the Fact table

In [0]:
fact_gl_transactions = (
    silver_df
    .withColumn(
        "transaction_key",
        F.concat_ws(
            "_",
            "company_code",
            "fiscal_year",
            "document_number",
            "line_item"
        )
    )
    .withColumn(
        "debit_amount",
        F.when(F.col("debit_credit") == "D", F.col("amount"))
         .otherwise(F.lit(0))
    )
    .withColumn(
        "credit_amount",
        F.when(F.col("debit_credit") == "C", F.col("amount"))
         .otherwise(F.lit(0))
    )
    .select(
        "transaction_key",
        "company_code",
        "fiscal_year",
        "document_number",
        "line_item",
        "posting_date",
        "document_date",
        "gl_account",
        "debit_credit",
        "amount",
        "debit_amount",
        "credit_amount",
        "currency",
        "vendor_id",
        "customer_id",
        "cost_center",
        "profit_center",
        "document_type",
        "reference",
        "description"
    )
)

print("Fact records:", fact_gl_transactions.count())


7.7 Create monthly financial summary

In [0]:
monthly_summary = (
    fact_gl_transactions
    .withColumn(
        "posting_month",
        F.date_format("posting_date", "yyyy-MM")
    )
    .groupBy(
        "company_code",
        "fiscal_year",
        "posting_month",
        "currency"
    )
    .agg(
        F.count("*").alias("total_line_items"),
        F.countDistinct("document_number").alias("total_journal_entries"),
        F.sum("debit_amount").alias("total_debit"),
        F.sum("credit_amount").alias("total_credit"),
        F.sum("amount").alias("total_posting_amount")
    )
    .withColumn(
        "debit_credit_difference",
        F.col("total_debit") - F.col("total_credit")
    )
)

display(monthly_summary)


7.8 Create GL account balances

In [0]:
account_balances = (
    fact_gl_transactions
    .groupBy(
        "company_code",
        "gl_account",
        "currency",
        "fiscal_year"
    )
    .agg(
        F.sum("debit_amount").alias("total_debit"),
        F.sum("credit_amount").alias("total_credit"),
        F.count("*").alias("transaction_count")
    )
    .withColumn(
        "net_debit_balance",
        F.col("total_debit") - F.col("total_credit")
    )
)

display(account_balances)


7.9 Save all Gold tables

In [0]:
gold_tables = {
    "dim_company": dim_company,
    "dim_gl_account": dim_gl_account,
    "dim_vendor": dim_vendor,
    "dim_customer": dim_customer,
    "fact_gl_transactions": fact_gl_transactions,
    "monthly_financial_summary": monthly_summary,
    "gl_account_balances": account_balances
}

for table_name, dataframe in gold_tables.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"ask_sarthak.gold.{table_name}"
        )
    )

print("All Gold tables created successfully!")


check the fact table:


In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM ask_sarthak.gold.fact_gl_transactions;

In [0]:
%sql
SELECT *
FROM ask_sarthak.gold.monthly_financial_summary
ORDER BY fiscal_year, posting_month
LIMIT 20;

In [0]:
%sql
COMMENT ON TABLE ask_sarthak.gold.fact_gl_transactions
IS 'Finance fact table containing individual general ledger journal line items, debit and credit amounts, company, GL account, posting date and financial references.';

COMMENT ON TABLE ask_sarthak.gold.dim_company
IS 'Company master dimension containing company codes and company names.';

COMMENT ON TABLE ask_sarthak.gold.dim_gl_account
IS 'General Ledger account dimension containing GL account codes, names and account classifications.';

COMMENT ON TABLE ask_sarthak.gold.dim_vendor
IS 'Vendor master dimension used to identify vendors associated with financial transactions.';

COMMENT ON TABLE ask_sarthak.gold.dim_customer
IS 'Customer master dimension used to identify customers associated with financial transactions.';

COMMENT ON TABLE ask_sarthak.gold.monthly_financial_summary
IS 'Monthly financial summary aggregated by company, fiscal year and currency.';

COMMENT ON TABLE ask_sarthak.gold.gl_account_balances
IS 'Aggregated debit, credit and net debit balances by company, GL account and fiscal year.';

In [0]:
%sql
SELECT
    table_name,
    comment
FROM ask_sarthak.information_schema.tables
WHERE table_schema = 'gold'
ORDER BY table_name;